# 🚀 ETF Backtesting - Simple Version

**Notebook นี้แก้ปัญหาทั้งหมดแล้ว - Run cell ไหนก่อนก็ได้!**

## วิธีใช้:
1. ⚠️ ตรวจสอบว่า **MySQL เปิดอยู่**
2. แก้ password ใน Step 1 ให้ตรง
3. Run cell ที่ต้องการ (ไม่ต้อง run ตามลำดับ)

---

## Step 1: ตั้งค่า Database (แก้ password ตรงนี้!)

In [ ]:
# ⚠️ แก้ password ตรงนี้ให้ตรงกับ MySQL ของคุณ
DB_CONFIG = {
    'host': '127.0.0.1',
    'user': 'root',
    'password': 'krittanut123456',  # ← แก้ตรงนี้
    'database': 'etf_backtesting',
    'port': 3306
}

print("✓ Database config loaded")
print(f"  Host: {DB_CONFIG['host']}:{DB_CONFIG['port']}")
print(f"  User: {DB_CONFIG['user']}")
print(f"  Database: {DB_CONFIG['database']}")

## Step 2: ทดสอบการเชื่อมต่อ MySQL

In [ ]:
import mysql.connector

try:
    conn = mysql.connector.connect(
        host=DB_CONFIG['host'],
        port=DB_CONFIG['port'],
        user=DB_CONFIG['user'],
        password=DB_CONFIG['password']
    )
    cursor = conn.cursor()
    cursor.execute("SELECT VERSION()")
    version = cursor.fetchone()[0]
    cursor.close()
    conn.close()
    
    print("✅ MySQL Connected!")
    print(f"   Version: {version}")
    
except Exception as e:
    print("❌ MySQL Connection Failed!")
    print(f"   Error: {e}")
    print("\n💡 ตรวจสอบ:")
    print("   1. MySQL เปิดอยู่หรือไม่? (XAMPP, MySQL Workbench)")
    print("   2. Password ถูกต้องหรือไม่? (แก้ที่ Step 1)")

## Step 3: Load Helper Functions

Cell นี้มี functions ที่จำเป็นทั้งหมด ไม่ต้อง import modules ภายนอก

In [ ]:
import mysql.connector
import pandas as pd
from datetime import datetime

def get_connection():
    """สร้าง database connection"""
    return mysql.connector.connect(**DB_CONFIG)

def show_portfolios():
    """แสดง portfolios ทั้งหมด"""
    conn = get_connection()
    query = """
    SELECT 
        p.portfolio_id,
        p.name,
        p.description,
        COUNT(pe.ticker) as num_etfs,
        p.created_at
    FROM portfolios p
    LEFT JOIN portfolio_etfs pe ON p.portfolio_id = pe.portfolio_id
    GROUP BY p.portfolio_id
    ORDER BY p.portfolio_id
    """
    df = pd.read_sql(query, conn)
    conn.close()
    return df

def show_portfolio_details(portfolio_id):
    """แสดงรายละเอียด portfolio"""
    conn = get_connection()
    
    # Portfolio info
    query1 = f"SELECT * FROM portfolios WHERE portfolio_id = {portfolio_id}"
    portfolio = pd.read_sql(query1, conn)
    
    # ETFs in portfolio
    query2 = f"""
    SELECT 
        pe.ticker,
        e.name,
        e.category,
        pe.weight
    FROM portfolio_etfs pe
    JOIN etfs e ON pe.ticker = e.ticker
    WHERE pe.portfolio_id = {portfolio_id}
    ORDER BY pe.weight DESC
    """
    etfs = pd.read_sql(query2, conn)
    conn.close()
    
    print(f"\n{'='*60}")
    print(f"Portfolio: {portfolio['name'].values[0]}")
    print(f"Description: {portfolio['description'].values[0]}")
    print(f"{'='*60}\n")
    
    return etfs

def show_etfs():
    """แสดง ETFs ทั้งหมด"""
    conn = get_connection()
    query = "SELECT * FROM etfs ORDER BY ticker"
    df = pd.read_sql(query, conn)
    conn.close()
    return df

def show_price_data(ticker, limit=10):
    """แสดงข้อมูลราคา"""
    conn = get_connection()
    query = f"""
    SELECT * FROM daily_prices 
    WHERE ticker = '{ticker}'
    ORDER BY date DESC
    LIMIT {limit}
    """
    df = pd.read_sql(query, conn)
    conn.close()
    return df

def count_records():
    """นับจำนวน records"""
    conn = get_connection()
    cursor = conn.cursor()
    
    tables = ['etfs', 'daily_prices', 'portfolios', 'backtests']
    results = {}
    
    for table in tables:
        cursor.execute(f"SELECT COUNT(*) FROM {table}")
        results[table] = cursor.fetchone()[0]
    
    cursor.close()
    conn.close()
    
    print("\n📊 Database Statistics:")
    print("=" * 40)
    for table, count in results.items():
        print(f"  {table:20s}: {count:,}")
    print("=" * 40)
    
    return results

print("✅ Helper functions loaded!")
print("\nAvailable functions:")
print("  • show_portfolios()")
print("  • show_portfolio_details(portfolio_id)")
print("  • show_etfs()")
print("  • show_price_data(ticker, limit=10)")
print("  • count_records()")

---

# 📊 ตอนนี้ใช้งานได้แล้ว!

Run cells ด้านล่างตามที่ต้องการ (ไม่ต้องเรียงลำดับ)

---

## ✅ ตรวจสอบข้อมูลในระบบ

In [ ]:
# นับจำนวน records
stats = count_records()

## 📁 ดู Portfolios ทั้งหมด

In [ ]:
# แสดง portfolios ทั้งหมด
portfolios_df = show_portfolios()
display(portfolios_df)

## 🔍 ดูรายละเอียด Portfolio

In [ ]:
# ดู portfolio ID 1 (Conservative 60/40)
portfolio_details = show_portfolio_details(1)
display(portfolio_details)

In [ ]:
# ดู portfolio ID 3 (Aggressive 90/10)
portfolio_details = show_portfolio_details(3)
display(portfolio_details)

## 📊 ดู ETFs ทั้งหมด

In [ ]:
# แสดง ETFs ทั้งหมด
etfs_df = show_etfs()
display(etfs_df.head(10))
print(f"\nTotal ETFs: {len(etfs_df)}")

## 💹 ดูข้อมูลราคา

In [ ]:
# ดูข้อมูลราคา SPY (10 วันล่าสุด)
price_df = show_price_data('SPY', limit=10)
display(price_df)

In [ ]:
# ดูข้อมูลราคา QQQ (20 วันล่าสุด)
price_df = show_price_data('QQQ', limit=20)
display(price_df)

## 📈 วิเคราะห์ข้อมูลด้วย Pandas

In [ ]:
# กรอง ETFs ตาม category
etfs_df = show_etfs()

# แสดงจำนวนแต่ละ category
category_counts = etfs_df['category'].value_counts()
print("\n📊 ETFs by Category:")
print("=" * 40)
for category, count in category_counts.items():
    print(f"  {category:30s}: {count}")
print("=" * 40)

In [ ]:
# ดูเฉพาะ US Equity ETFs
us_equity = etfs_df[etfs_df['category'] == 'US Equity']
print("\n🇺🇸 US Equity ETFs:")
display(us_equity[['ticker', 'name', 'expense_ratio']])

In [ ]:
# ดูเฉพาะ Bond ETFs
bonds = etfs_df[etfs_df['category'].str.contains('Bond', na=False)]
print("\n💰 Bond ETFs:")
display(bonds[['ticker', 'name', 'category', 'expense_ratio']])

## 📊 วิเคราะห์ราคาย้อนหลัง

In [ ]:
# ดูราคา SPY ปี 2024
conn = get_connection()
query = """
SELECT 
    date,
    close,
    volume
FROM daily_prices
WHERE ticker = 'SPY'
  AND date >= '2024-01-01'
ORDER BY date DESC
LIMIT 20
"""
spy_2024 = pd.read_sql(query, conn)
conn.close()

print("\n📈 SPY Price in 2024 (Latest 20 days):")
display(spy_2024)

In [ ]:
# เปรียบเทียบราคาเฉลี่ย 2024 ของหลาย ETFs
conn = get_connection()
query = """
SELECT 
    ticker,
    AVG(close) as avg_price,
    MIN(close) as min_price,
    MAX(close) as max_price,
    COUNT(*) as trading_days
FROM daily_prices
WHERE date >= '2024-01-01'
  AND ticker IN ('SPY', 'QQQ', 'VOO', 'VTI', 'IWM')
GROUP BY ticker
ORDER BY avg_price DESC
"""
comparison = pd.read_sql(query, conn)
conn.close()

print("\n📊 Major ETFs Comparison (2024):")
display(comparison)

## 📈 Simple Price Chart

In [ ]:
import matplotlib.pyplot as plt

# ดึงข้อมูลราคา SPY ปี 2024
conn = get_connection()
query = """
SELECT date, close
FROM daily_prices
WHERE ticker = 'SPY'
  AND date >= '2024-01-01'
ORDER BY date
"""
spy_data = pd.read_sql(query, conn)
conn.close()

# สร้าง chart
plt.figure(figsize=(12, 6))
plt.plot(spy_data['date'], spy_data['close'], linewidth=2, color='blue')
plt.title('SPY Price Chart (2024)', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price ($)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"\n📊 Total trading days: {len(spy_data)}")
print(f"💰 Price range: ${spy_data['close'].min():.2f} - ${spy_data['close'].max():.2f}")
print(f"📈 Latest price: ${spy_data['close'].iloc[-1]:.2f}")

---

# 🎉 สรุป

Notebook นี้แสดงวิธีใช้งานพื้นฐาน:
- ✅ เชื่อมต่อ MySQL
- ✅ ดูข้อมูล Portfolios
- ✅ ดูข้อมูล ETFs
- ✅ ดูข้อมูลราคา
- ✅ วิเคราะห์ข้อมูลด้วย Pandas
- ✅ สร้าง charts

---

## 💡 ต่อไป:

ถ้าต้องการใช้ **Analytics และ Backtesting** แบบเต็มรูปแบบ:
1. ใช้ `jupyter_interface.py` functions
2. ดู `ETF_Backtesting_Notebook.ipynb` สำหรับตัวอย่างเต็ม
3. อ่าน `USER_GUIDE_TH.md` สำหรับคู่มือฉบับสมบูรณ์

---